# 1.1 - Data Engineering - UNIPLU Preparation and Event Extraction

Este notebook realiza a extração e discretização de eventos de precipitação subdiários a partir dos dados brutos compactados do UNIPLU (`src/data/UNIPLU/*.zip`), salvando os registros em formato **Parquet** (`src/5 - Results/events_table_{year}.parquet`).

**Metodologia:**
- Leitura direta dos arquivos Parquet compactados do UNIPLU com PyArrow;
- Discretização de eventos contínuos utilizando múltiplos MITs (30 min, 6h, 24h);
- Filtro de lâmina mínima de 1.0 mm.

In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload
%autoreload 2

import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

# Root setup
PROJECT_ROOT = Path.cwd() if Path.cwd().name == 'src' else Path.cwd() / 'src'
sys.path.insert(0, str(PROJECT_ROOT.parent))

from src.config import ROOT, UNIPLU_DIR, MITS_MINUTES, MINIMUM_DEPTH_MM
from src.data.loader import load_gauge_info, load_rainfall_data
from src.events.detection import determine_rainfall_events, compute_rainfall_time_step
from src.utils.logger import save_result

RESULTS_DIR = ROOT / 'src' / '5 - Results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('UNIPLU Dir:', UNIPLU_DIR)
print('Results Dir:', RESULTS_DIR)

## 1. Carregamento dos Metadados das Estações Pluviométricas

In [ ]:
df_info = load_gauge_info()
print('Total de estações cadastradas:', len(df_info))
print('Redes de monitoramento:')
print(df_info['network'].value_counts())
df_info.head()

## 2. Parâmetros de Discretização de Eventos

In [ ]:
TARGET_YEARS = [2023, 2024]
TARGET_STATES = ['SP']  # Altere para os estados desejados ou None para todos
SAMPLE_STATIONS = 50   # Amostra para processamento rápido (ou None para todas)

print('Anos:', TARGET_YEARS)
print('Estados:', TARGET_STATES)
print('MITs:', MITS_MINUTES)

## 3. Discretização de Eventos por Estação e por MIT

In [ ]:
all_events_list = []

for year in TARGET_YEARS:
    print(f'=== Processando Ano {year} ===')
    df_rain = load_rainfall_data(
        states=TARGET_STATES,
        years=[year],
        sample=SAMPLE_STATIONS,
        seed=42
    )
    if df_rain.empty:
        print('Sem dados para o ano', year)
        continue
    print(f'Registros: {len(df_rain):,} | Estações: {df_rain["gauge_code"].nunique()}')

    for code_val, group in df_rain.groupby('gauge_code'):
        group_sorted = group.sort_values('datetime').reset_index(drop=True)
        if len(group_sorted) < 2:
            continue
        time_step = compute_rainfall_time_step(group_sorted)
        rain_pos = group_sorted[group_sorted['rain_mm'] > 0].reset_index(drop=True)
        if len(rain_pos) < 2:
            continue
        for mit_val in MITS_MINUTES:
            mit_td = pd.Timedelta(minutes=mit_val)
            events = determine_rainfall_events(mit_td, rain_pos, time_step)
            if not events.empty:
                events = events[events['rain_mm'] >= MINIMUM_DEPTH_MM].copy()
                events['year'] = year
                events['mit_minutes'] = mit_val
                all_events_list.append(events)

if all_events_list:
    df_events = pd.concat(all_events_list, ignore_index=True)
    print(f'Total de eventos detectados: {len(df_events):,}')
else:
    df_events = pd.DataFrame()
    print('Nenhum evento detectado.')

## 4. Persistência dos Eventos em Parquet

In [ ]:
if not df_events.empty:
    out_file = save_result(df_events, 'events_table.parquet', subdir='5 - Results')
    print('Salvo em:', out_file)
    print(df_events.head())